# 06 — SHAP Explainability
**EduForecast** | Model Interpretability & Decision Support | Notebook 6 of 6

In [ ]:
import sys
import warnings
import random
from datetime import date
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
import shap
import torch
import torch.nn as nn
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
shap.initjs()

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import PROCESSED_DIR, FIGURES_DIR, METRICS_DIR, RESULTS_DIR, RANDOM_SEED

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

sns.set_style('whitegrid')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
DPI = 150

print(f'SHAP version : {shap.__version__}')
print(f'PyTorch      : {torch.__version__}')
print('Environment ready.')

## 1. Load Models and Data

In [ ]:
# ── Static feature datasets ───────────────────────────────────────────────
train_static = pd.read_csv(PROCESSED_DIR / 'train_static.csv')
test_static  = pd.read_csv(PROCESSED_DIR / 'test_static.csv')

DROP_COLS   = ['id_student', 'code_module', 'code_presentation', 'is_at_risk']
FEATURE_COLS = [c for c in train_static.columns if c not in DROP_COLS]

X_train_df = train_static[FEATURE_COLS]
X_test_df  = test_static[FEATURE_COLS]
y_train    = train_static['is_at_risk'].values
y_test     = test_static['is_at_risk'].values

X_train_np = X_train_df.values.astype(np.float32)
X_test_np  = X_test_df.values.astype(np.float32)

print(f'Train : {X_train_np.shape}  |  Test : {X_test_np.shape}')
print(f'Features ({len(FEATURE_COLS)}): {FEATURE_COLS[:5]} ... {FEATURE_COLS[-3:]}')

In [ ]:
# ── Retrain Random Forest (same params as notebook 03) ────────────────────
print('Training Random Forest...')
scaler_rf = StandardScaler()
X_train_sc = scaler_rf.fit_transform(X_train_np)
X_test_sc  = scaler_rf.transform(X_test_np)

rf = RandomForestClassifier(
    n_estimators=200, max_depth=10,
    random_state=RANDOM_SEED,
    class_weight='balanced', n_jobs=-1,
)
rf.fit(X_train_sc, y_train)
rf_probs = rf.predict_proba(X_test_sc)[:, 1]
rf_preds = (rf_probs >= 0.5).astype(int)
print(f'RF retrained. Test AUC check: ', end='')
from sklearn.metrics import roc_auc_score
print(f'{roc_auc_score(y_test, rf_probs):.4f}  (should match notebook 03: 0.9472)')

# Store probabilities on full test_static for the risk report
test_static = test_static.copy()
test_static['rf_prob'] = rf_probs
test_static['rf_pred'] = rf_preds
print('Models loaded and ready.')

## 2. SHAP for Random Forest

In [ ]:
print('Computing SHAP values via TreeExplainer (full test set)...')

tree_exp = shap.TreeExplainer(rf)
# SHAP 0.43+: returns ndarray (n_samples, n_features, n_classes)
shap_3d   = tree_exp.shap_values(X_test_sc)          # (n, p, 2)
shap_rf   = shap_3d[:, :, 1]                         # (n, p) — class 1 = At-Risk
base_val  = float(tree_exp.expected_value[1])

print(f'SHAP values shape (class 1): {shap_rf.shape}')
print(f'Base value (expected log-odds, class 1): {base_val:.4f}')

# Build Explanation object for waterfall / newer plots
rf_explanation = shap.Explanation(
    values       = shap_rf,
    base_values  = np.full(len(shap_rf), base_val),
    data         = X_test_sc,
    feature_names= FEATURE_COLS,
)

In [ ]:
# ── Plot 1: Beeswarm — global feature importance ──────────────────────────
shap.summary_plot(
    shap_rf, X_test_df,
    max_display=15, show=False,
    plot_type='dot',
)
fig = plt.gcf()
fig.set_size_inches(10, 7)
plt.title('SHAP Global Feature Importance — Random Forest (At-Risk Prediction)',
          fontsize=12, pad=10)
plt.tight_layout()
out1 = FIGURES_DIR / '06_shap_rf_beeswarm.png'
fig.savefig(out1, dpi=DPI, bbox_inches='tight')
plt.close(fig)
print(f'Saved → {out1}')

In [ ]:
# ── Plot 2: Bar — mean |SHAP| ─────────────────────────────────────────────
shap.summary_plot(
    shap_rf, X_test_df,
    max_display=15, show=False,
    plot_type='bar',
)
fig = plt.gcf()
fig.set_size_inches(9, 7)
plt.title('Mean |SHAP| Values — Random Forest', fontsize=12, pad=10)
plt.tight_layout()
out2 = FIGURES_DIR / '06_shap_rf_bar.png'
fig.savefig(out2, dpi=DPI, bbox_inches='tight')
plt.close(fig)
print(f'Saved → {out2}')

# Print top-10 features by mean |SHAP|
mean_abs_shap = np.abs(shap_rf).mean(axis=0)
rf_shap_df = pd.DataFrame({'feature': FEATURE_COLS, 'mean_abs_shap': mean_abs_shap})
rf_shap_df = rf_shap_df.sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

print(f'\n{"Rank":<5} {"Feature":<32} {"Mean|SHAP|":>12}')
print('-' * 52)
for i, row in rf_shap_df.head(10).iterrows():
    print(f'{i+1:<5} {row["feature"]:<32} {row["mean_abs_shap"]:>12.6f}')

In [ ]:
# ── Local explanations: select 3 students ────────────────────────────────
idx_high = int(np.argmax(rf_probs))                          # highest risk
idx_low  = int(np.argmin(rf_probs))                          # lowest risk
# Borderline: closest to 0.5
idx_mid  = int(np.argmin(np.abs(rf_probs - 0.5)))

for label, idx in [('High Risk', idx_high), ('Borderline', idx_mid), ('Low Risk', idx_low)]:
    prob     = rf_probs[idx]
    true_lbl = y_test[idx]
    sv_i     = shap_rf[idx]                                  # (p,)
    pos_top3 = np.argsort(sv_i)[-3:][::-1]                  # push toward at-risk
    neg_top3 = np.argsort(sv_i)[:3]                         # push away from at-risk
    print(f'\n── {label} (test idx={idx}) ──────────────────────────────────')
    print(f'  Predicted probability : {prob:.4f}  |  True label: {"At-Risk" if true_lbl else "Not At-Risk"}')
    print(f'  Top 3 factors pushing TOWARD risk:')
    for k in pos_top3:
        print(f'    {FEATURE_COLS[k]:<30} SHAP = {sv_i[k]:+.6f}')
    print(f'  Top 3 factors REDUCING risk:')
    for k in neg_top3:
        print(f'    {FEATURE_COLS[k]:<30} SHAP = {sv_i[k]:+.6f}')

In [ ]:
# ── Plots 3–5: Waterfall plots — High / Borderline / Low risk ─────────────
for label, idx, fname in [
    ('High Risk Student',        idx_high, '06_shap_waterfall_high.png'),
    ('Borderline Student',       idx_mid,  '06_shap_waterfall_mid.png'),
    ('Low Risk Student',         idx_low,  '06_shap_waterfall_low.png'),
]:
    shap.plots.waterfall(rf_explanation[idx], max_display=12, show=False)
    fig = plt.gcf()
    fig.set_size_inches(12, 7)
    fig.suptitle(f'SHAP Waterfall — {label}  (RF P(at-risk)={rf_probs[idx]:.3f})',
                 fontsize=11, y=1.01)
    out = FIGURES_DIR / fname
    fig.savefig(out, dpi=DPI, bbox_inches='tight')
    plt.close(fig)
    print(f'Saved → {out}')

## 3. SHAP for TFT-Inspired Model

In [ ]:
# ── Redefine TemporalAttentionModel (from notebook 05) ───────────────────
class TemporalAttentionModel(nn.Module):
    def __init__(self, input_size=7, hidden_size=64, num_layers=2,
                 num_heads=4, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Sequential(nn.Linear(input_size, hidden_size), nn.ReLU())
        self.lstm       = nn.LSTM(hidden_size, hidden_size, num_layers,
                                  batch_first=True,
                                  dropout=dropout if num_layers > 1 else 0.0)
        self.attention  = nn.MultiheadAttention(hidden_size, num_heads,
                                                dropout=dropout, batch_first=True)
        self.layer_norm = nn.LayerNorm(hidden_size)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(hidden_size // 2, 1),
        )

    def forward(self, x, key_padding_mask=None, return_attention=False):
        x_proj   = self.input_proj(x)
        lstm_out, _ = self.lstm(x_proj)
        attn_out, attn_w = self.attention(
            lstm_out, lstm_out, lstm_out,
            key_padding_mask=key_padding_mask,
            need_weights=True, average_attn_weights=True,
        )
        norm_out = self.layer_norm(attn_out + lstm_out)
        if key_padding_mask is not None:
            valid  = (~key_padding_mask).float().unsqueeze(-1)
            pooled = (norm_out * valid).sum(1) / valid.sum(1).clamp(min=1)
        else:
            pooled = norm_out.mean(1)
        logits = self.classifier(pooled).squeeze(-1)
        return (logits, attn_w) if return_attention else logits


tft_model = TemporalAttentionModel(input_size=7)
tft_model.load_state_dict(
    torch.load(RESULTS_DIR / 'tft_best_model.pth', map_location='cpu')
)
tft_model.eval()
print('TFT model loaded.')
print(f'Parameters: {sum(p.numel() for p in tft_model.parameters()):,}')

In [ ]:
# ── Build synthetic-sequence mapper static → sequential ──────────────────
# Sequential feature order: weekly_clicks, is_active, cumulative_clicks,
#   click_trend, weeks_since_active, assessment_score, assessment_submitted
N_WEEKS = 40
N_SEQ   = 7

IDX = {
    'total_clicks':             FEATURE_COLS.index('total_clicks'),
    'active_weeks':             FEATURE_COLS.index('active_weeks'),
    'mean_weekly_clicks':       FEATURE_COLS.index('mean_weekly_clicks'),
    'mean_score':               FEATURE_COLS.index('mean_score'),
    'num_assessments_submitted':FEATURE_COLS.index('num_assessments_submitted'),
}

def static_to_seq(X_np: np.ndarray) -> np.ndarray:
    """Map static feature matrix (n, 32) → synthetic sequence (n, 40, 7)."""
    n = X_np.shape[0]
    seq = np.zeros((n, N_WEEKS, N_SEQ), dtype=np.float32)

    mean_wk  = X_np[:, IDX['mean_weekly_clicks']]    # avg clicks per active week
    act_rate = np.clip(X_np[:, IDX['active_weeks']] / N_WEEKS, 0, 1)
    score    = X_np[:, IDX['mean_score']]
    n_assess = X_np[:, IDX['num_assessments_submitted']]
    total_cl = X_np[:, IDX['total_clicks']]

    week_idx = np.arange(1, N_WEEKS + 1, dtype=np.float32)[None, :]  # (1, 40)

    seq[:, :, 0] = mean_wk[:, None]                    # weekly_clicks
    seq[:, :, 1] = act_rate[:, None]                   # is_active
    seq[:, :, 2] = total_cl[:, None] * week_idx / N_WEEKS  # cumulative (linear)
    seq[:, :, 3] = 0.0                                 # click_trend
    seq[:, :, 4] = 0.0                                 # weeks_since_active
    seq[:, :, 5] = score[:, None]                      # assessment_score
    seq[:, :, 6] = (n_assess[:, None] / 20.0).clip(0, 1)  # assessment_submitted
    return seq

# Fit scaler on synthetic training sequences
train_seqs = static_to_seq(X_train_np)                # (n_train, 40, 7)
seq_scaler = StandardScaler()
seq_scaler.fit(train_seqs.reshape(-1, N_SEQ))
print('Synthetic-sequence scaler fitted on training data.')

def tft_predict_static(X_np: np.ndarray) -> np.ndarray:
    """Static features → TFT at-risk probability."""
    seqs = static_to_seq(X_np)                        # (n, 40, 7)
    seqs_sc = seq_scaler.transform(
        seqs.reshape(-1, N_SEQ)
    ).reshape(-1, N_WEEKS, N_SEQ).astype(np.float32)

    probs: list[float] = []
    tft_model.eval()
    with torch.no_grad():
        for s in range(0, len(seqs_sc), 128):
            batch = torch.from_numpy(seqs_sc[s:s + 128])
            p = torch.sigmoid(tft_model(batch)).numpy()
            probs.extend(p.tolist())
    return np.array(probs, dtype=np.float32)

# Quick sanity check
sample_prob = tft_predict_static(X_test_np[:5])
print(f'Sanity check — TFT probs on 5 test samples: {sample_prob.round(4)}')

In [ ]:
# ── KernelExplainer: background=50, explain=50 ───────────────────────────
print('Setting up KernelExplainer (background=50 samples)...')

rng = np.random.RandomState(RANDOM_SEED)
bg_idx   = rng.choice(len(X_train_np), size=50, replace=False)
test_idx = rng.choice(len(X_test_np),  size=50, replace=False)

X_bg      = X_train_np[bg_idx]
X_explain = X_test_np[test_idx]
y_explain = y_test[test_idx]

kernel_exp = shap.KernelExplainer(tft_predict_static, X_bg)
print('Computing SHAP values (nsamples=100, may take ~3 minutes)...')
shap_tft = kernel_exp.shap_values(X_explain, nsamples=100)   # (50, 32)

print(f'TFT SHAP values shape: {np.array(shap_tft).shape}')

In [ ]:
# ── Plot 6: TFT SHAP bar — mean |SHAP| ───────────────────────────────────
tft_shap_arr = np.array(shap_tft)    # (50, 32)

shap.summary_plot(
    tft_shap_arr,
    pd.DataFrame(X_explain, columns=FEATURE_COLS),
    max_display=15, show=False, plot_type='bar',
)
fig = plt.gcf()
fig.set_size_inches(9, 7)
plt.title('Mean |SHAP| Values — TFT-Inspired Model', fontsize=12, pad=10)
plt.tight_layout()
out6 = FIGURES_DIR / '06_shap_tft_bar.png'
fig.savefig(out6, dpi=DPI, bbox_inches='tight')
plt.close(fig)
print(f'Saved → {out6}')

# Top 10 TFT features
mean_abs_tft = np.abs(tft_shap_arr).mean(axis=0)
tft_shap_df = pd.DataFrame({'feature': FEATURE_COLS, 'mean_abs_shap': mean_abs_tft})
tft_shap_df = tft_shap_df.sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

print(f'\nTop 10 TFT features by mean |SHAP|:')
print(f'{"Rank":<5} {"Feature":<32} {"Mean|SHAP|":>12}')
print('-' * 52)
for i, row in tft_shap_df.head(10).iterrows():
    print(f'{i+1:<5} {row["feature"]:<32} {row["mean_abs_shap"]:>12.6f}')

## 4. SHAP Comparison: RF vs TFT

In [ ]:
# Rank-based comparison (lower rank = more important)
rf_shap_df['rf_rank']  = rf_shap_df.index + 1
tft_shap_df['tft_rank'] = tft_shap_df.index + 1

comparison = rf_shap_df[['feature', 'mean_abs_shap', 'rf_rank']].merge(
    tft_shap_df[['feature', 'mean_abs_shap', 'tft_rank']],
    on='feature', suffixes=('_rf', '_tft'),
)
comparison['rank_diff'] = (comparison['rf_rank'] - comparison['tft_rank']).abs()
comparison = comparison.sort_values('rf_rank').reset_index(drop=True)

top10_feats = comparison.head(10)['feature'].tolist()
agree_top3  = set(rf_shap_df.head(3)['feature']) & set(tft_shap_df.head(3)['feature'])
disagree    = comparison.nlargest(3, 'rank_diff')[['feature','rf_rank','tft_rank','rank_diff']]

print('Top 10 features — RF vs TFT rank comparison:')
print(f'{"Feature":<32} {"RF Rank":>8} {"TFT Rank":>9} {"|Diff|":>8}')
print('-' * 60)
for _, row in comparison.head(10).iterrows():
    print(f'{row["feature"]:<32} {row["rf_rank"]:>8.0f} {row["tft_rank"]:>9.0f} {row["rank_diff"]:>8.0f}')

print(f'\nFeatures where RF and TFT agree (top 3): {agree_top3 if agree_top3 else "None"}')
print(f'\nFeatures with largest rank disagreement:')
print(disagree.to_string(index=False))

In [ ]:
# ── Plot 7: Side-by-side RF vs TFT SHAP for top 10 features ──────────────
top10 = comparison.head(10).copy()
# Normalise to [0,1] for fair visual comparison
top10['rf_norm']  = top10['mean_abs_shap_rf']  / top10['mean_abs_shap_rf'].max()
top10['tft_norm'] = top10['mean_abs_shap_tft'] / top10['mean_abs_shap_tft'].max()

x      = np.arange(len(top10))
width  = 0.38
labels = [f.replace('region_', 'reg.') for f in top10['feature']]

fig, ax = plt.subplots(figsize=(13, 6))
bars_rf  = ax.bar(x - width/2, top10['rf_norm'],  width, label='Random Forest',   color='#e74c3c', alpha=0.85)
bars_tft = ax.bar(x + width/2, top10['tft_norm'], width, label='TFT-Inspired',    color='#3498db', alpha=0.85)

for bars in (bars_rf, bars_tft):
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.01, f'{h:.2f}',
                ha='center', va='bottom', fontsize=7.5)

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Normalised Mean |SHAP| (1.0 = max)')
ax.set_ylim(0, 1.18)
ax.set_title('Feature Importance Comparison: RF vs TFT-Inspired (SHAP)', fontsize=13, pad=12)
ax.legend(fontsize=10)
plt.tight_layout()
out7 = FIGURES_DIR / '06_shap_comparison.png'
fig.savefig(out7, dpi=DPI)
plt.close(fig)
print(f'Saved → {out7}')

## 5. Management Decision Report

In [ ]:
top_feature  = rf_shap_df.iloc[0]['feature']
top2_feature = rf_shap_df.iloc[1]['feature']

# Per-student top contributing feature
test_top_feat = [FEATURE_COLS[i] for i in np.argmax(shap_rf, axis=1)]
test_static['top_shap_feature'] = test_top_feat

# Risk tiers
high_risk = (rf_probs >= 0.70).sum()
med_risk  = ((rf_probs >= 0.40) & (rf_probs < 0.70)).sum()
low_risk  = (rf_probs < 0.40).sum()

# Top 10 at-risk students
top10_students = (
    test_static.nlargest(10, 'rf_prob')
    [['id_student', 'rf_prob', 'is_at_risk', 'top_shap_feature']]
    .reset_index(drop=True)
)

today = date.today().isoformat()

print('=' * 60)
print('EDUFORECAST — STUDENT RISK REPORT')
print('=' * 60)
print(f'Report date  : {today}')
print(f'Model        : Random Forest (AUC: 0.9472)')
print(f'Test cohort  : {len(test_static):,} students')
print()
print('TOP 10 AT-RISK STUDENTS')
print('-' * 60)
for _, row in top10_students.iterrows():
    actual = 'WITHDRAWN' if row['is_at_risk'] else 'active'
    print(f"Student {int(row['id_student']):>8} | "
          f"Risk: {row['rf_prob']:.1%} | "
          f"Top factor: {row['top_shap_feature']:<22} | "
          f"Actual: {actual}")
print()
print('KEY SYSTEM FINDINGS')
print('-' * 60)
print(f'Most important predictor  : {top_feature}')
print(f'Second predictor          : {top2_feature}')
print(f'Students flagged HIGH RISK (>70%): {high_risk:,}')
print(f'Students flagged MED  RISK (40-70%): {med_risk:,}')
print(f'Students flagged LOW  RISK (<40%): {low_risk:,}')
print('=' * 60)

In [ ]:
# Save enriched report to results/final_report.md
tft_top = tft_shap_df.iloc[0]['feature']
agree_yn = 'Yes' if agree_top3 else 'No'

report_md = f"""# EduForecast \u2014 Final Results Report

**Generated:** {today}  
**Project:** EduForecast: Unified AI Early Warning and Decision Support System

---

## 1. Executive Summary

EduForecast predicts student withdrawal risk from OULAD interaction data.
Four models were trained and compared. The Random Forest achieves the highest
overall AUC (0.9472), while the TFT-inspired model provides the best
early-warning capability (AUC 0.759 at week 5).

## 2. Dataset

- Source: OULAD (Kuzilek et al., 2017)
- Total registrations: 32,593
- Unique students: 28,785
- Withdrawal rate: 31.2%

## 3. Model Comparison

| Model | AUC | F1 | Accuracy | Sensitivity |
|-------|-----|----|----------|-------------|
| Logistic Regression | 0.9375 | 0.8596 | 0.8560 | 0.8900 |
| Random Forest | 0.9472 | 0.8629 | 0.8584 | 0.9489 |
| LSTM | 0.9436 | 0.8611 | 0.8564 | 0.9578 |
| TFT-Inspired | 0.9433 | 0.8619 | 0.8573 | 0.9519 |

## 4. Temporal AUC (Early Warning)

| Week | TFT AUC | LSTM AUC |
|------|---------|----------|
| 5  | 0.759 | 0.429 |
| 10 | 0.838 | 0.287 |
| 20 | 0.901 | 0.356 |
| 39 | 0.916 | 0.902 |

## 5. SHAP Explainability

**Top predictors (Random Forest SHAP):**

| Rank | Feature | Mean |SHAP| |
|------|---------|------|
""" + "".join(
    f"| {i+1} | {row['feature']} | {row['mean_abs_shap']:.6f} |\n"
    for i, row in rf_shap_df.head(5).iterrows()
) + f"""
**Top TFT predictor:** {tft_top}  
**RF vs TFT agreement on top 3:** {agree_yn}  

## 6. Student Risk Report ({today})

| Student ID | Predicted Risk | Top Factor | Actual Outcome |
|-----------|---------------|------------|----------------|
""" + "".join(
    f"| {int(r['id_student'])} | {r['rf_prob']:.1%} | "
    f"{r['top_shap_feature']} | {'Withdrawn' if r['is_at_risk'] else 'Active'} |\n"
    for _, r in top10_students.iterrows()
) + f"""
**Students flagged HIGH RISK (>70%):** {high_risk}  
**Students flagged MED  RISK (40-70%):** {med_risk}  
**Students flagged LOW  RISK (<40%):** {low_risk}  

## 7. References

Kuzilek, J., Hlosta, M., & Zdrahal, Z. (2017). Open University Learning Analytics dataset.
*Scientific Data*, 4, 170171. https://doi.org/10.1038/sdata.2017.171
"""

out_report = RESULTS_DIR / 'final_report.md'
out_report.write_text(report_md, encoding='utf-8')
print(f'Report saved \u2192 {out_report}')

## 6. Save All SHAP Figures — Summary

In [ ]:
print('\nFigures generated in this notebook:')
saved = sorted(FIGURES_DIR.glob('06_shap_*.png'))
for f in saved:
    sz = f.stat().st_size // 1024
    print(f'  {f.name:<40} ({sz} KB)')

tft_top_feat = tft_shap_df.iloc[0]['feature']
agree_str    = 'Yes' if agree_top3 else 'No'

print()
print('=' * 50)
print('SHAP ANALYSIS COMPLETE')
print('=' * 50)
print(f'RF SHAP   : computed on {len(X_test_np):,} test registrations')
print(f'TFT SHAP  : computed on 50 samples (KernelExplainer)')
print(f'Top predictor (RF)  : {rf_shap_df.iloc[0]["feature"]}')
print(f'Top predictor (TFT) : {tft_top_feat}')
print(f'Agreement on top 3  : {agree_str}')
print(f'Figures saved       : {len(saved)}')
print('=' * 50)